# Neural Network for Classification

In this notebook, we will create a simple fully connected neural network using Pytorch. We will evaluate the quality of the model when trained on the three different datasets used in the logistic regression Notebook.

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import DataLoader
from utils import shuffle_data_df, train_test_split_df, standardise

sns.set_style("whitegrid")

df = pd.read_csv("../Data/c_w_data.csv")
df = df.copy().drop(columns=['Score'])


df.head()

,Series,Surface,Round,dj_pts,dj_rank,dj_odds,opp_pts,opp_rank,opp_odds,dj_win,age,is_outdoor,is_five_sets,5SMA
0,International,Clay,2nd Round,417,97,5.00,1170,28,1.14,0,18,1,0,1.000000
1,Masters,Hard,1st Round,440,97,3.50,1385,18,1.28,0,18,1,0,0.500000
2,Grand Slam,Hard,1st Round,431,97,2.50,825,43,1.50,1,18,1,1,0.333333
3,Grand Slam,Hard,2nd Round,431,97,3.50,1230,24,1.28,1,18,1,1,0.500000
4,Grand Slam,Hard,3rd Round,431,97,2.75,770,48,1.39,0,18,1,1,0.600000


In [2]:
# Shuffle data
df_shuff = shuffle_data_df(df)
df_shuff.head()

,Series,Surface,Round,dj_pts,dj_rank,dj_odds,opp_pts,opp_rank,opp_odds,dj_win,age,is_outdoor,is_five_sets,5SMA
1057,Masters 1000,Clay,The Final,8260,1,1.440,5750,5,2.75,1,35,1,0,0.8
938,Grand Slam,Grass,The Final,12415,1,1.550,6620,3,2.60,1,32,1,1,1.0
1228,ATP250,Hard,Quarterfinals,4580,5,1.170,1120,47,5.00,1,38,0,0,0.8
683,Grand Slam,Clay,1st Round,13845,1,1.002,603,87,34.00,1,28,1,1,1.0
282,ATP500,Hard,The Final,7330,4,1.360,2195,15,3.20,1,22,1,0,0.8


In [3]:
# Train test split
df_train, df_test = train_test_split_df(df_shuff, 0.8)

In [4]:
# Forming dataset
df_odds_train = df_train.copy()[['dj_odds', 'opp_odds', 'dj_win']]
df_odds_test = df_test.copy()[['dj_odds', 'opp_odds', 'dj_win']]

In [5]:
# Separating target from predictors
y_train, y_test = df_odds_train['dj_win'].to_numpy().reshape(-1, 1), df_odds_test['dj_win'].to_numpy().reshape(-1, 1)
X_train, X_test = df_odds_train[['dj_odds', 'opp_odds']].to_numpy(), df_odds_test[['dj_odds', 'opp_odds']].to_numpy()

In [6]:
# Standardise and augment prediction matrix
X_train_std = standardise(X_train)
X_test_std = standardise(X_test, X_train)

X_train_sa = np.concatenate([np.ones((X_train_std.shape[0], 1)), X_train_std], axis=1)
X_test_sa = np.concatenate([np.ones((X_test_std.shape[0], 1)), X_test_std], axis=1)

In [7]:
# Forming dataset
df_nodds_train = df_train.copy().drop(columns=['dj_odds', 'opp_odds'])
df_nodds_test = df_test.copy().drop(columns=['dj_odds', 'opp_odds'])

In [8]:
# Form one-hot data
series = pd.get_dummies(df_nodds_train['Series'])
surface = pd.get_dummies(df_nodds_train['Surface'])
round = pd.get_dummies(df_nodds_train['Round'])
one_hots_train = pd.concat([series, surface, round], axis=1).astype(int)
one_hots_train.head()

,ATP250,ATP500,Grand Slam,International,International Gold,Masters,Masters 1000,Masters Cup,Carpet,Clay,Grass,Hard,1st Round,2nd Round,3rd Round,4th Round,Quarterfinals,Round Robin,Semifinals,The Final
1057,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1
938,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1
1228,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0
683,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0
282,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1


In [9]:
# Same for test set
series = pd.get_dummies(df_nodds_test['Series'])
surface = pd.get_dummies(df_nodds_test['Surface'])
round = pd.get_dummies(df_nodds_test['Round'])
one_hots_test = pd.concat([series, surface, round], axis=1).astype(int)

In [10]:
# Drop old columns
df_nodds_train = df_nodds_train.copy().drop(columns=['Series', 'Surface', 'Round'])
df_nodds_test = df_nodds_test.copy().drop(columns=['Series', 'Surface', 'Round'])

In [11]:
# Separate target from predictors
X_train, X_test = df_nodds_train.drop(columns='dj_win'), df_nodds_test.drop(columns='dj_win')

In [12]:
# Now convert to numpy arrays, standardise, and concatenate with dummy columns (dont want to standardise
# one hot encoded columns)
X_nodds_train_std = standardise(X_train.to_numpy())
X_nodds_test_std = standardise(X_test.to_numpy(), scaler=X_train.to_numpy())

X_nodds_train_final = np.concatenate([np.ones((X_nodds_train_std.shape[0], 1)), 
                                    X_nodds_train_std, one_hots_train.to_numpy()], axis=1)
X_nodds_test_final = np.concatenate([np.ones((X_nodds_test_std.shape[0], 1)), 
                                    X_nodds_test_std, one_hots_test.to_numpy()],axis=1)

In [13]:
# Form one-hot data
series = pd.get_dummies(df_train['Series'])
surface = pd.get_dummies(df_train['Surface'])
round = pd.get_dummies(df_train['Round'])
one_hots_train = pd.concat([series, surface, round], axis=1).astype(int)
one_hots_train.head()

,ATP250,ATP500,Grand Slam,International,International Gold,Masters,Masters 1000,Masters Cup,Carpet,Clay,Grass,Hard,1st Round,2nd Round,3rd Round,4th Round,Quarterfinals,Round Robin,Semifinals,The Final
1057,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1
938,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1
1228,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0
683,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0
282,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1


In [14]:
# Same for test set
series = pd.get_dummies(df_test['Series'])
surface = pd.get_dummies(df_test['Surface'])
round = pd.get_dummies(df_test['Round'])
one_hots_test = pd.concat([series, surface, round], axis=1).astype(int)

In [15]:
# Drop old columns
df_train = df_train.copy().drop(columns=['Series', 'Surface', 'Round'])
df_test = df_test.copy().drop(columns=['Series', 'Surface', 'Round'])

In [16]:
# Separate target from predictors
X_train, X_test = df_train.drop(columns='dj_win'), df_test.drop(columns='dj_win')

In [27]:
# Now convert to numpy arrays, standardise, and concatenate with dummy columns (dont want to standardise
# one hot encoded columns)
X_full_train_std = standardise(X_train.to_numpy())
X_full_test_std = standardise(X_test.to_numpy(), scaler=X_train.to_numpy())

X_full_train_final = np.concatenate([np.ones((X_full_train_std.shape[0], 1)), 
                                    X_full_train_std, one_hots_train.to_numpy()], axis=1)
X_full_test_final = np.concatenate([np.ones((X_full_test_std.shape[0], 1)), 
                                    X_full_test_std, one_hots_test.to_numpy()], axis=1)

In [28]:
odds_trainloader = DataLoader(X_train_std, batch_size=32)
odds_testloader = DataLoader(X_test_std, batch_size=32)

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(features, 512),
            nn.ReLU(),
            nn.Linear(512, 2))

    def forward(self, X):
        logits = self.layer(X)



In [25]:
model = NeuralNetwork(2)
print(model)

NeuralNetwork(
  (layer): Sequential(
    (0): Linear(in_features=2, out_features=4, bias=True)
    (1): ReLU()
    (2): Linear(in_features=4, out_features=2, bias=True)
  )
)


In [ ]:
def train(dataloader, modelm, loss, optim):

    for batch, (X, y)